In [3]:
# THIS CELL FOR GETTING SBIR COMPANY DATA

import requests
import sqlite3
import time
import json

# API base URL
BASE_URL = "https://api.www.sbir.gov/public/api/firm"

# Database setup
DB_NAME = "server/companies.db"

# Function to create the database tables
def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Create awards table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS companies (
            firm_nid INTEGER PRIMARY KEY,
            company_name TEXT,
            sbir_url TEXT,
            uei TEXT,
            duns TEXT,
            address1 TEXT,
            address2 TEXT,
            city TEXT,
            state TEXT,
            zip TEXT,
            company_url TEXT,
            hubzone_owned TEXT,
            socially_economically_disadvantaged TEXT,
            woman_owned TEXT,
            number_awards INTEGER
        )
    """)

    conn.commit()
    conn.close()

# Function to fetch data from API
def fetch_data(page):
    params = {
        "rows": 50,
        "start": page * 50,
        #"name": "laser",
        #"uei": "integer_value",
        #'sort': 'name', #state, #uei
        }
    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching page {page}: {response.status_code}")
        return None

# Function to insert data into SQLite database
def insert_data(data):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    for item in data:
        cursor.execute("""
            INSERT OR REPLACE INTO companies (
                firm_nid,
                company_name,
                sbir_url,
                uei,
                duns,
                address1,
                address2,
                city,
                state,
                zip,
                company_url,
                hubzone_owned,
                socially_economically_disadvantaged,
                woman_owned,
                number_awards
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            item.get("firm_nid"),
            item.get("company_name"),
            item.get("sbir_url"),
            item.get("uei"),
            item.get("duns"),
            item.get("address1"), 
            item.get("address2"),
            item.get("city"),
            item.get("state"),
            item.get("zip"),
            item.get("company_url"),
            item.get("hubzone_owned"),
            item.get("socially_economically_disadvantaged"),
            item.get("women_owned"),
            item.get("number_awards")
        ))    
    conn.commit()
    conn.close()

# Main execution
def main():
    setup_database()
    
    
    #for page in range(3):  #number of pages
    page = 0 #around 4300 total pages
    while True:
        print(f"Fetching page {page + 1}...")
        data = fetch_data(page)
        if not data or len(data) == 0:
            print("No more data")
            break
        insert_data(data)
        time.sleep(0.1)  # Avoid excessive requests
        page += 1

    print("Company data successfully stored in SQLite database.")

if __name__ == "__main__":
    main()


Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Fetching page 5...
Fetching page 6...
Fetching page 7...
Fetching page 8...
Fetching page 9...
Fetching page 10...
Fetching page 11...
Fetching page 12...
Fetching page 13...
Fetching page 14...
Fetching page 15...
Fetching page 16...
Fetching page 17...
Fetching page 18...
Fetching page 19...
Fetching page 20...
Fetching page 21...
Fetching page 22...
Fetching page 23...
Fetching page 24...
Fetching page 25...
Fetching page 26...
Fetching page 27...
Fetching page 28...
Fetching page 29...
Fetching page 30...
Fetching page 31...
Fetching page 32...
Fetching page 33...
Fetching page 34...
Fetching page 35...
Fetching page 36...
Fetching page 37...
Fetching page 38...
Fetching page 39...
Fetching page 40...
Fetching page 41...
Fetching page 42...
Fetching page 43...
Fetching page 44...
Fetching page 45...
Fetching page 46...
Fetching page 47...
Fetching page 48...
Fetching page 49...
Fetching page 50...
Fetching 

In [4]:
def check_schema():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='companies'")
    print("\nCompanies table schema:")
    print(cursor.fetchone()[0])
    
    conn.close()

def fetch_companies():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM companies") # LIMIT 10")
    rows = cursor.fetchall()
    
    for row in rows:
        print(row)
    
    conn.close()

def count_entries():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    # Count entries in awards table
    cursor.execute("SELECT COUNT(*) FROM companies")
    companies_count = cursor.fetchone()[0]
    
    conn.close()
    
    return {
        "companies": companies_count,
    }  

if __name__ == "__main__":
    check_schema()
    
    print("Companies:")
    fetch_companies()

    counts = count_entries()
    print(f"Number of companies: {counts['companies']}")


Companies table schema:
CREATE TABLE companies (
            firm_nid INTEGER PRIMARY KEY,
            company_name TEXT,
            sbir_url TEXT,
            uei TEXT,
            duns TEXT,
            address1 TEXT,
            address2 TEXT,
            city TEXT,
            state TEXT,
            zip TEXT,
            company_url TEXT,
            hubzone_owned TEXT,
            socially_economically_disadvantaged TEXT,
            woman_owned TEXT,
            number_awards INTEGER
        )
Companies:
(12330, 'Savari Inc.', 'https://www.sbir.gov/portfolio/12330', 'N4L2JXCCN751', '826394657', '2005 De La Cruz Blvd.', 'ste. 131', 'Santa Clara', 'CA', '95050-', None, 'No', 'Yes', None, 8)
(12332, 'HEMOSONICS', 'https://www.sbir.gov/portfolio/12332', 'PFFKCTPB2NX2', '193921041', '400 Preston Avenue', 'Suite 250', 'CHARLOTTESVILLE', 'VA', '22903-4585', 'www.hemosonics.com', 'No', 'No', None, 11)
(12344, 'NESS ENGINEERING, INC.', 'https://www.sbir.gov/portfolio/12344', 'EQNQKG3KL